# OHLC Conditional — CSV-based Analysis

Loads pre-generated samples from CSV files produced by `generate_samples.py`
and compares the generated Close paths against ground-truth Close.

No model inference is performed here — just load, parse, and analyse.

Expected CSVs (under `<GEN_DIR>/<CHECKPOINT_STEM>/`):
- `train_generated_close.csv` — generated Close paths for training windows
- `train_gt_ohlc.csv`         — ground-truth OHLC context for training windows
- `val_generated_close.csv`   — generated Close paths for validation windows
- `val_gt_ohlc.csv`           — ground-truth OHLC context for validation windows

CSV schemas
-----------
`*_generated_close.csv`: `window_idx | sample_idx | file | window_start | step_000 … step_{L-1}`

`*_gt_ohlc.csv`: `window_idx | feature | file | window_start | step_000 … step_{L-1}`

In [41]:
# ── USER INPUTS ───────────────────────────────────────────────────────────────
CHECKPOINT_STEM = (
    "NO_NO_FAKE_FTS_PROC_CLOS_ep-212_20260412"
)  # <-- sub-folder name under GEN_DIR (= checkpoint filename without .pt)

GEN_DIR = "../data/generated/conditional"  # base directory where generate_samples.py writes

SPLIT   = "val"   # "train" | "val" | "both"  — which split(s) to analyse
# ─────────────────────────────────────────────────────────────────────────────

In [42]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import acf

ckpt_dir = os.path.join(GEN_DIR, CHECKPOINT_STEM)
assert os.path.isdir(ckpt_dir), f"Directory not found: {ckpt_dir}"
print(f"Reading from: {ckpt_dir}")

Reading from: ../data/generated/conditional\NO_NO_FAKE_FTS_PROC_CLOS_ep-212_20260412


## 1. Load CSVs

In [43]:
def load_split(split, ckpt_dir):
    """
    Load one split's generated-close and GT-OHLC CSVs.

    Returns
    -------
    df_gen  : DataFrame  (window_idx, sample_idx, file, window_start, step_*)
    df_ohlc : DataFrame  (window_idx, feature, file, window_start, step_*)
    step_cols : list of step column names
    """
    gen_path  = os.path.join(ckpt_dir, f"{split}_generated_close.csv")
    ohlc_path = os.path.join(ckpt_dir, f"{split}_gt_ohlc.csv")

    assert os.path.isfile(gen_path),  f"Not found: {gen_path}"
    assert os.path.isfile(ohlc_path), f"Not found: {ohlc_path}"

    df_gen  = pd.read_csv(gen_path)
    df_ohlc = pd.read_csv(ohlc_path)

    step_cols = [c for c in df_gen.columns if c.startswith("step_")]
    seq_len   = len(step_cols)

    n_windows = df_gen["window_idx"].nunique()
    n_samples = df_gen["sample_idx"].nunique()
    features  = df_ohlc["feature"].unique().tolist()

    print(f"[{split}] windows={n_windows}  samples/window={n_samples}  "
          f"seq_len={seq_len}  features={features}")

    return df_gen, df_ohlc, step_cols


splits_to_run = ["train", "val"] if SPLIT == "both" else [SPLIT]
data = {}

for split in splits_to_run:
    try:
        df_gen, df_ohlc, step_cols = load_split(split, ckpt_dir)
        data[split] = (df_gen, df_ohlc, step_cols)
    except AssertionError as e:
        print(f"Skipping {split}: {e}")

[val] windows=40  samples/window=30  seq_len=64  features=['open', 'high', 'low', 'close']


## 2. Parse into NumPy arrays

Reshape DataFrames into `(n_windows, n_samples, seq_len)` for generated Close
and `(n_windows, seq_len)` per OHLC feature for ground-truth context.

In [44]:
def parse_arrays(df_gen, df_ohlc, step_cols):
    """
    Returns
    -------
    gen_3d   : (n_windows, n_samples, seq_len)  — generated Close
    gt_close : (n_windows, seq_len)             — ground-truth Close
    gt_open  : (n_windows, seq_len)
    gt_high  : (n_windows, seq_len)
    gt_low   : (n_windows, seq_len)
    window_ids : list of window_idx values (sorted)
    """
    window_ids = sorted(df_gen["window_idx"].unique())
    n_windows  = len(window_ids)
    n_samples  = df_gen["sample_idx"].nunique()
    seq_len    = len(step_cols)

    gen_3d = np.zeros((n_windows, n_samples, seq_len), dtype=np.float32)
    for wi, wid in enumerate(window_ids):
        sub = df_gen[df_gen["window_idx"] == wid].sort_values("sample_idx")
        gen_3d[wi] = sub[step_cols].to_numpy(dtype=np.float32)

    def gt_feat(feat_name):
        sub = df_ohlc[df_ohlc["feature"] == feat_name].sort_values("window_idx")
        return sub[step_cols].to_numpy(dtype=np.float32)   # (n_windows, seq_len)

    gt_open  = gt_feat("open")
    gt_high  = gt_feat("high")
    gt_low   = gt_feat("low")
    gt_close = gt_feat("close")

    return gen_3d, gt_close, gt_open, gt_high, gt_low, window_ids


parsed = {}
for split, (df_gen, df_ohlc, step_cols) in data.items():
    parsed[split] = parse_arrays(df_gen, df_ohlc, step_cols)
    gen_3d, gt_close, gt_open, gt_high, gt_low, window_ids = parsed[split]
    print(f"[{split}] gen_3d={gen_3d.shape}  gt_close={gt_close.shape}")

[val] gen_3d=(40, 30, 64)  gt_close=(40, 64)


---
## Analyses

The cells below run on the **first available split** by default.  
Change `ANALYSIS_SPLIT` to switch between `"train"` / `"val"`.

In [45]:
ANALYSIS_SPLIT = splits_to_run[0]   # change to "val" if needed

gen_3d, gt_close, gt_open, gt_high, gt_low, window_ids = parsed[ANALYSIS_SPLIT]
n_windows, n_samples, seq_len = gen_3d.shape
steps = np.arange(seq_len)

# Flat 2-D views for distribution-level comparisons
all_gen_close = gen_3d.reshape(n_windows * n_samples, seq_len)   # (N*B, L)
all_gt_close  = np.repeat(gt_close, n_samples, axis=0)           # (N*B, L)
all_gt_high   = np.repeat(gt_high,  n_samples, axis=0)
all_gt_low    = np.repeat(gt_low,   n_samples, axis=0)

print(f"Split         : {ANALYSIS_SPLIT}")
print(f"n_windows     : {n_windows}")
print(f"n_samples     : {n_samples}")
print(f"seq_len       : {seq_len}")
print(f"all_gen_close : {all_gen_close.shape}")

Split         : val
n_windows     : 40
n_samples     : 30
seq_len       : 64
all_gen_close : (1200, 64)


## 3. Sample path overlays

For each shown window: ground-truth OHL band + GT Close (top row) vs
GT OHL band + all generated Close samples (bottom row).

In [46]:
N_SHOW  = min(n_windows, 4)   # number of windows to print individually

for w in range(N_SHOW):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

    for ax in axes:
        ax.fill_between(steps, gt_low[w], gt_high[w],
                        alpha=0.15, color="gray", label="[L, H] band")
        ax.plot(steps, gt_open[w],  color="steelblue", lw=1.0, ls="--", alpha=0.7, label="Open")
        ax.plot(steps, gt_high[w],  color="green",     lw=1.0, ls="--", alpha=0.7, label="High")
        ax.plot(steps, gt_low[w],   color="red",       lw=1.0, ls="--", alpha=0.7, label="Low")
        ax.set_xlabel("step")
        ax.set_ylabel("normalised price")

    # Left panel: ground-truth Close
    axes[0].plot(steps, gt_close[w], color="black", lw=1.8, label="GT Close")
    axes[0].set_title(f"Window {window_ids[w]} — Ground Truth Close")
    axes[0].legend(fontsize=8, loc="upper left")

    # Right panel: all generated Close samples
    for s in range(n_samples):
        axes[1].plot(steps, gen_3d[w, s], color="tab:orange", lw=0.9, alpha=0.5)
    axes[1].set_title(f"Window {window_ids[w]} — Generated Close ({n_samples} samples)")

    plt.suptitle(
        f"[{ANALYSIS_SPLIT}] Window {window_ids[w]} — OHL context + generated/GT Close",
        y=1.02, fontsize=12
    )
    plt.tight_layout()
    plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13060\2234422535.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. OHLC constraint satisfaction

Checks whether the generated Close satisfies `L ≤ Close ≤ H`.

> Note: data are normalised column-wise, so this constraint can be broken locally even for perfectly generated samples — treat the rate as a diagnostic, not an absolute criterion.

In [47]:
# ── Generated Close constraint ────────────────────────────────────────────────
above_low  = all_gen_close >= all_gt_low
below_high = all_gen_close <= all_gt_high
in_band    = above_low & below_high

step_rate   = in_band.mean(axis=0)
fully_valid = in_band.all(axis=1).mean()
stepwise    = in_band.mean()

# ── Ground-truth Close constraint (baseline) ──────────────────────────────────
gt_above_low  = all_gt_close >= all_gt_low
gt_below_high = all_gt_close <= all_gt_high
gt_in_band    = gt_above_low & gt_below_high

gt_step_rate   = gt_in_band.mean(axis=0)
gt_fully_valid = gt_in_band.all(axis=1).mean()
gt_stepwise    = gt_in_band.mean()

print("=" * 55)
print("  OHLC CONSTRAINT  L ≤ Close ≤ H")
print("=" * 55)
print(f"{'Metric':<35} {'Generated':>10} {'GT (ref)':>10}")
print("-" * 55)
print(f"{'Step-wise satisfaction rate':<35} {stepwise:>10.4f} {gt_stepwise:>10.4f}")
print(f"{'Fully valid samples':<35} {fully_valid:>10.4f} {gt_fully_valid:>10.4f}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(steps, step_rate,    color="tab:orange", lw=1.5, label=f"Generated  (overall={stepwise:.3f})")
ax.plot(steps, gt_step_rate, color="tab:blue",   lw=1.5, label=f"GT (ref)   (overall={gt_stepwise:.3f})", ls="--")
ax.axhline(1.0, color="black", lw=0.8, ls=":", label="Perfect (1.0)")
ax.set_ylim(0, 1.05)
ax.set_xlabel("step t")
ax.set_ylabel("fraction in [L, H]")
ax.set_title(f"[{ANALYSIS_SPLIT}] OHLC constraint per step — Generated vs GT")
ax.legend()
plt.tight_layout()
plt.show()

  OHLC CONSTRAINT  L ≤ Close ≤ H
Metric                               Generated   GT (ref)
-------------------------------------------------------
Step-wise satisfaction rate             0.1170     0.1156
Fully valid samples                     0.0000     0.0000


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13060\2796722365.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Marginal distribution of Close values
We are testing:
* Marginal close distribution overall
* QQ plots for quantile values
* ECDF (Empirical CDF), or F(x)=(y<=x) / N, where y is the number of values smaller or equal to x and N is the total number of observations.

In [48]:
gen_flat = all_gen_close.ravel()
gt_flat  = all_gt_close.ravel()

ks_stat, ks_p = scipy_stats.ks_2samp(gen_flat, gt_flat)

def moments(x, label):
    return {
        "label"           : label,
        "mean"            : np.mean(x),
        "std"             : np.std(x),
        "skewness"        : scipy_stats.skew(x),
        "excess_kurtosis" : scipy_stats.kurtosis(x, fisher=True),
        "q01"             : np.quantile(x, 0.01),
        "q99"             : np.quantile(x, 0.99),
    }

df_moments = pd.DataFrame([
    moments(gen_flat, "Generated Close"),
    moments(gt_flat,  "Ground-truth Close"),
]).set_index("label").round(5)

print(f"KS statistic = {ks_stat:.4f}  |  p-value = {ks_p:.4e}")
print()
display(df_moments)

combined = np.concatenate([gen_flat, gt_flat])
lo, hi   = np.quantile(combined, [0.0005, 0.9995])
bins     = np.linspace(lo, hi, 60)
probs    = np.linspace(0.01, 0.99, min(len(gen_flat), 5_000))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.hist(gt_flat,  bins=bins, density=True, alpha=0.5, label="GT Close",        color="tab:blue")
ax.hist(gen_flat, bins=bins, density=True, alpha=0.5, label="Generated Close", color="tab:orange")
ax.set_title("Marginal Close distribution")
ax.set_xlabel("Close"); ax.set_ylabel("density"); ax.legend(fontsize=8)

ax = axes[1]
q_gt  = np.quantile(gt_flat,  probs)
q_gen = np.quantile(gen_flat, probs)
ax.scatter(q_gt, q_gen, s=4, alpha=0.5, color="steelblue")
lims = [min(q_gt.min(), q_gen.min()), max(q_gt.max(), q_gen.max())]
ax.plot(lims, lims, "r--", lw=1.2, label="y = x (perfect)")
ax.set_title("QQ plot — Generated vs GT Close")
ax.set_xlabel("GT quantiles"); ax.set_ylabel("Generated quantiles"); ax.legend(fontsize=8)

ax = axes[2]
for vals, label, color in [(gt_flat, "GT Close", "tab:blue"), (gen_flat, "Generated", "tab:orange")]:
    s = np.sort(vals)
    ax.plot(s, np.arange(1, len(s) + 1) / len(s), label=label, lw=1.2)
ax.set_xlim(lo, hi)
ax.set_title(f"ECDF  (KS={ks_stat:.3f}, p={ks_p:.2e})")
ax.set_xlabel("Close"); ax.set_ylabel("CDF"); ax.legend(fontsize=8)

plt.suptitle(f"[{ANALYSIS_SPLIT}] Marginal distribution of Close", y=1.02)
plt.tight_layout()
plt.show()

KS statistic = 0.0069  |  p-value = 5.2066e-02



,mean,std,skewness,excess_kurtosis,q01,q99
label,,,,,,
Generated Close,-0.00229,0.99676,-0.03482,-0.75324,-2.05808,2.00051
Ground-truth Close,0.00000,1.00000,-0.02628,-0.71793,-2.10410,2.02574


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13060\1838549286.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Close increments distribution and ACF

In [49]:
gen_inc = np.diff(all_gen_close, axis=1).ravel()
gt_inc  = np.diff(all_gt_close,  axis=1).ravel()

ks_inc, ks_p_inc = scipy_stats.ks_2samp(gen_inc, gt_inc)
print(f"Increment KS = {ks_inc:.4f}  p = {ks_p_inc:.4e}")

df_inc = pd.DataFrame([
    moments(gen_inc, "Generated ΔClose"),
    moments(gt_inc,  "GT ΔClose"),
]).set_index("label").round(5)
display(df_inc)

N_ACF    = min(50, n_windows)
LAGS     = min(30, seq_len // 2)
lag_axis = np.arange(1, LAGS + 1)

def mean_acf(paths, lags, use_diff=False):
    acfs = []
    for p in paths[:N_ACF]:
        x = np.diff(p) if use_diff else p
        acfs.append(acf(x, nlags=lags, fft=True)[1:])
    return np.mean(acfs, axis=0)

# For ACF we pick one sample per window (sample 0) to avoid repetition artefacts
gen_paths_s0 = gen_3d[:, 0, :]   # (n_windows, seq_len)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))

lo_i, hi_i = np.quantile(np.concatenate([gen_inc, gt_inc]), [0.005, 0.995])
bins_i = np.linspace(lo_i, hi_i, 60)
axes[0, 0].hist(gt_inc,  bins=bins_i, density=True, alpha=0.5, label="GT ΔClose",        color="tab:blue")
axes[0, 0].hist(gen_inc, bins=bins_i, density=True, alpha=0.5, label="Generated ΔClose", color="tab:orange")
axes[0, 0].set_title(f"Increment marginal  (KS={ks_inc:.3f}, p={ks_p_inc:.2e})")
axes[0, 0].set_xlabel("ΔClose"); axes[0, 0].legend(fontsize=8)

probs_i = np.linspace(0.01, 0.99, min(len(gen_inc), 5_000))
q_gt_i  = np.quantile(gt_inc,  probs_i)
q_gen_i = np.quantile(gen_inc, probs_i)
axes[0, 1].scatter(q_gt_i, q_gen_i, s=4, alpha=0.5, color="steelblue")
lims_i = [min(q_gt_i.min(), q_gen_i.min()), max(q_gt_i.max(), q_gen_i.max())]
axes[0, 1].plot(lims_i, lims_i, "r--", lw=1.2)
axes[0, 1].set_title("QQ — Generated ΔClose vs GT ΔClose")
axes[0, 1].set_xlabel("GT quantiles"); axes[0, 1].set_ylabel("Generated quantiles")

axes[1, 0].plot(lag_axis, mean_acf(gt_close,      LAGS, use_diff=False), label="GT Close",        color="tab:blue")
axes[1, 0].plot(lag_axis, mean_acf(gen_paths_s0,  LAGS, use_diff=False), label="Generated Close", color="tab:orange")
axes[1, 0].axhline(0, color="black", lw=0.8, ls="--")
axes[1, 0].set_title("ACF of Close levels")
axes[1, 0].set_xlabel("lag"); axes[1, 0].set_ylabel("autocorrelation"); axes[1, 0].legend(fontsize=8)

axes[1, 1].plot(lag_axis, mean_acf(gt_close,      LAGS, use_diff=True), label="GT ΔClose",        color="tab:blue")
axes[1, 1].plot(lag_axis, mean_acf(gen_paths_s0,  LAGS, use_diff=True), label="Generated ΔClose", color="tab:orange")
axes[1, 1].axhline(0, color="black", lw=0.8, ls="--")
axes[1, 1].set_title("ACF of Close increments")
axes[1, 1].set_xlabel("lag"); axes[1, 1].set_ylabel("autocorrelation"); axes[1, 1].legend(fontsize=8)

plt.suptitle(f"[{ANALYSIS_SPLIT}] Increment distribution and autocorrelation", y=1.01)
plt.tight_layout()
plt.show()

Increment KS = 0.0125  p = 1.3631e-05


,mean,std,skewness,excess_kurtosis,q01,q99
label,,,,,,
Generated ΔClose,-0.00001,0.36816,-0.06266,0.81675,-0.94525,0.90351
GT ΔClose,0.00019,0.37597,0.00209,0.91522,-0.95522,0.96162


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13060\4229105845.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Per-step conditional mean ± std

At each timestep: generated sample mean vs GT Close, with ±1 std band.

In [50]:
gen_mean = gen_3d.mean(axis=1)   # (n_windows, seq_len)
gen_std  = gen_3d.std(axis=1)    # (n_windows, seq_len)

N_SHOW2 = min(n_windows, 4)
fig, axes = plt.subplots(1, N_SHOW2, figsize=(5 * N_SHOW2, 4), sharey=False)
if N_SHOW2 == 1:
    axes = [axes]

for col, w in enumerate(range(N_SHOW2)):
    ax = axes[col]
    ax.plot(steps, gt_close[w],  color="black",      lw=1.5, label="GT Close",       zorder=3)
    ax.plot(steps, gen_mean[w],  color="tab:orange", lw=1.2, label="Generated mean", zorder=2)
    ax.fill_between(
        steps,
        gen_mean[w] - gen_std[w],
        gen_mean[w] + gen_std[w],
        alpha=0.3, color="tab:orange", label="±1 std"
    )
    ax.fill_between(steps, gt_low[w], gt_high[w], alpha=0.10, color="gray", label="[L, H]")
    ax.set_title(f"Window {window_ids[w]}")
    ax.set_xlabel("step")
    ax.set_ylabel("normalised price")

axes[0].legend(fontsize=7, loc="upper left")
plt.suptitle(f"[{ANALYSIS_SPLIT}] Per-step conditional mean ± std of generated Close", y=1.02)
plt.tight_layout()
plt.show()

mae  = np.abs(gen_mean - gt_close).mean()
rmse = np.sqrt(((gen_mean - gt_close) ** 2).mean())
print(f"Mean absolute error  (generated mean vs GT): {mae:.5f}")
print(f"RMSE                 (generated mean vs GT): {rmse:.5f}")
print(f"Mean generated std   (residual uncertainty): {gen_std.mean():.5f}")

Mean absolute error  (generated mean vs GT): 0.08896
RMSE                 (generated mean vs GT): 0.11863
Mean generated std   (residual uncertainty): 0.10183


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13060\684449256.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Train vs Val comparison (requires SPLIT="both")

Side-by-side summary statistics for train and validation splits.

In [51]:
if len(parsed) < 2:
    print("Set SPLIT='both' at the top to enable train/val comparison.")
else:
    rows = []
    for sp, (g3d, gtc, gto, gth, gtl, wids) in parsed.items():
        nw, ns, sl = g3d.shape
        gen_all = g3d.reshape(nw * ns, sl)
        gt_all  = np.repeat(gtc, ns, axis=0)
        ks, ks_p  = scipy_stats.ks_2samp(gen_all.ravel(), gt_all.ravel())
        gm = g3d.mean(axis=1)
        mae  = np.abs(gm - gtc).mean()
        rmse = np.sqrt(((gm - gtc) ** 2).mean())
        in_b = ((gen_all >= np.repeat(gtl, ns, axis=0)) &
                (gen_all <= np.repeat(gth, ns, axis=0))).mean()
        rows.append({
            "split"          : sp,
            "n_windows"      : nw,
            "n_samples"      : ns,
            "KS stat"        : round(ks, 4),
            "KS p-value"     : f"{ks_p:.2e}",
            "MAE (mean vs GT)": round(mae, 5),
            "RMSE"           : round(rmse, 5),
            "OHLC step rate" : round(float(in_b), 4),
        })
    display(pd.DataFrame(rows).set_index("split"))

Set SPLIT='both' at the top to enable train/val comparison.


# 9. FTS metrics

In [52]:
import sys
sys.path.insert(0, "..")
import replication.stylized_facts as sf
from pathlib import Path

output_dir = Path("../images/stylized_facts_ohlc_conditional")
output_dir.mkdir(exist_ok=True, parents=True)


In [53]:
print(gt_close.shape, gt_close.shape[0]*gt_close.shape[1])
print(all_gen_close.shape, all_gen_close.shape[0]*all_gen_close.shape[1])

(40, 64) 2560
(1200, 64) 76800


In [54]:
# ── Flat 1-D arrays for sf.distribution ──────────────────────────────────────
# gt_close  : (n_windows, seq_len) → ravel to 1-D
# all_gen_close : (n_windows * n_samples, seq_len) — already defined above
gt_close_pooled = gt_close.ravel()        # shape: (n_windows * seq_len,)
gen_paths_arr   = all_gen_close.ravel()   # shape: (n_windows * n_samples * seq_len,)

print(gt_close_pooled.shape)
print(gen_paths_arr.shape)

# ── Object arrays of individual paths for sf.acf / sf.leverage_effect ────────
gt_close_obj = np.empty(n_windows, dtype=object)
for i, p in enumerate(gt_close):
    gt_close_obj[i] = p                   # each element: (seq_len,) GT Close window

gen_paths_obj = np.empty(len(all_gen_close), dtype=object)
for i, p in enumerate(all_gen_close):
    gen_paths_obj[i] = p                  # each element: (seq_len,) generated path

print(gt_close_obj.shape, gt_close_obj[0].shape)
print(gen_paths_obj.shape, gen_paths_obj[0].shape)

(2560,)
(76800,)
(40,) (64,)
(1200,) (64,)


In [55]:
sf.distribution(
    gt_close_pooled,
    file_name=str(output_dir / f"gt_distribution_{CHECKPOINT_STEM}"),
    scale="log",
    multiple=False,
    normalize=True,
    granuality=100,
)

sf.distribution(
    gen_paths_arr,
    file_name=str(output_dir / f"generated_distribution_{CHECKPOINT_STEM}"),
    scale="log",
    multiple=False,
    normalize=True,
    granuality=100,
)


In [56]:
sf.acf(
    gt_close_obj,
    file_name=str(output_dir / f"gt_volatility_clustering_{CHECKPOINT_STEM}"),
    for_abs=True,
    multiple=True,
    fit=False,
    scale="log",
    max_lag=seq_len // 2, # in the original it was 1000, but the seq_len was 2048
)

sf.acf(
    gen_paths_obj,
    file_name=str(output_dir / f"generated_volatility_clustering_{CHECKPOINT_STEM}"),
    for_abs=True,
    multiple=True,
    fit=False,
    scale="log",
    max_lag=seq_len // 2,
)


In [57]:
sf.leverage_effect(
    gt_close_obj,
    file_name=str(output_dir / f"gt_leverage_effect_{CHECKPOINT_STEM}"),
    multiple=True,
    min_lag=1,
    max_lag=min(100, seq_len // 2),
)

sf.leverage_effect(
    gen_paths_obj,
    file_name=str(output_dir / f"generated_leverage_effect_{CHECKPOINT_STEM}"),
    multiple=True,
    min_lag=1,
    max_lag=min(100, seq_len // 2),
)


array([-0.0320045 , -0.01633778, -0.00782033, -0.01161719, -0.01363013,
       -0.00206363,  0.01990975,  0.03269854,  0.04481162,  0.05349038,
        0.06564009,  0.07987395,  0.09338028,  0.08742388,  0.08907321,
        0.09512039,  0.10230669,  0.11301873,  0.11176494,  0.1065199 ,
        0.09523914,  0.08089962,  0.05938424,  0.02892629,  0.00867167,
       -0.00548314, -0.00954147, -0.00936562, -0.00841934, -0.00577574,
        0.00558725])

In [58]:
import powerlaw

def fit_powerlaw(returns, max_sample=15000, seed=42):
    x = np.abs(np.asarray(returns, dtype=float))
    x = x[np.isfinite(x) & (x > 0)]

    if len(x) > max_sample:
        rng_local = np.random.default_rng(seed)
        x = rng_local.choice(x, size=max_sample, replace=False)

    f = powerlaw.Fit(x, discrete=False, verbose=True,
                     parameter_ranges={"alpha": [1.5, 10.0]})

    x_tail = x[x >= f.power_law.xmin]
    alpha_mle = 1 + len(x_tail) / np.sum(np.log(x_tail / f.power_law.xmin))

    return {
        "alpha"    : f.power_law.alpha,
        "alpha_mle": alpha_mle,
        "xmin"     : f.power_law.xmin,
        "ks"       : f.power_law.D,
        "n_total"  : len(x),
        "n_tail"   : len(x_tail),
    }

# gt_close_pooled and gen_paths_arr are the flat 1-D arrays built in the previous block.
# They are already log-returns, so concatenation directly gives the return sample —
# identical to np.concatenate(list(paths)) in Training_vs_Generated_Analysis.
for arr, label in [(gt_close_pooled, "GT Close"), (gen_paths_arr, "Generated Close")]:
    r = fit_powerlaw(arr)
    print(f"{label}")
    print(f"  alpha (powerlaw) : {r['alpha']:.4f}")
    print(f"  alpha_mle        : {r['alpha_mle']:.4f}")
    print(f"  xmin             : {r['xmin']:.6f}")
    print(f"  KS distance      : {r['ks']:.4f}")
    print(f"  n_total          : {r['n_total']:,}")
    print(f"  n_tail           : {r['n_tail']:,}")
    print()


Calculating best minimal value for power law fit


Fitting xmin: 100%|██████████| 2558/2558 [00:04<00:00, 634.31it/s] 


GT Close
  alpha (powerlaw) : 7.2924
  alpha_mle        : 7.2924
  xmin             : 1.598633
  KS distance      : 0.0567
  n_total          : 2,560
  n_tail           : 245

Calculating best minimal value for power law fit


Fitting xmin: 100%|██████████| 14992/14992 [00:54<00:00, 277.41it/s]

Generated Close
  alpha (powerlaw) : 8.0090
  alpha_mle        : 8.0090
  xmin             : 1.678221
  KS distance      : 0.0360
  n_total          : 15,000
  n_tail           : 1,073



## Interpretation guide

| Test | Passing (overfit / well-conditioned) | Failing |
|------|--------------------------------------|---------|
| **Sample paths** (§3) | Generated Close tracks GT inside [L,H] | Paths wander outside [L,H] or uncorrelated with GT |
| **OHLC constraint** (§4) | Step-wise rate > 0.90 | Rate < 0.5 → model ignores OHL context |
| **Marginal KS** (§5) | KS p > 0.05; QQ on diagonal | S-curve in QQ (wrong tail shape) |
| **Increment KS** (§6) | KS p > 0.05; similar std | Generated std much larger → noise not removed |
| **Level ACF** (§6) | Generated ACF tracks GT shape | Flat or too fast decay → wrong temporal structure |
| **Increment ACF** (§6) | Both near 0 at all lags | Positive lags → spurious momentum |
| **Per-step mean** (§7) | MAE near 0; small residual std | Large MAE → bad conditioning; large std → excess uncertainty |

# Additional window-conditional analysis metrics

## 10. CRPS — probabilistic forecast quality

**CRPS** (Continuous Ranked Probability Score) evaluates the full predictive distribution, not just the posterior mean:

$$\text{CRPS}(\hat{F}, y) = \mathbb{E}[|X - y|] - \tfrac{1}{2}\,\mathbb{E}[|X - X'|]$$

where $X, X'$ are independent draws from the ensemble. Lower is better.  
A perfect *point* forecast collapses to MAE, so CRPS ≤ MAE always — the gap measures how much the spread helps beyond the mean prediction.

In [59]:
def crps_ensemble_batch(samples, obs):
    """
    CRPS for every (window, step) position.

    Parameters
    ----------
    samples : (n_windows, n_samples, seq_len)
    obs     : (n_windows, seq_len)

    Returns
    -------
    crps : (n_windows, seq_len)
    """
    # E[|X - y|] averaged over samples — shape (n_windows, seq_len)
    term1 = np.mean(np.abs(samples - obs[:, np.newaxis, :]), axis=1)

    # E[|X - X'|] via mean of all pairwise |x_i - x_j| — O(M^2) per window, fine for M=30
    n_windows_loc = samples.shape[0]
    term2 = np.zeros_like(term1)
    for w in range(n_windows_loc):
        s = samples[w]                                             # (M, L)
        diff = np.abs(s[:, np.newaxis, :] - s[np.newaxis, :, :])  # (M, M, L)
        term2[w] = diff.mean(axis=(0, 1))

    return term1 - 0.5 * term2


crps_grid   = crps_ensemble_batch(gen_3d, gt_close)  # (n_windows, seq_len)
mean_crps   = float(crps_grid.mean())
mean_mae    = float(np.abs(gen_3d.mean(axis=1) - gt_close).mean())

print(f"Mean CRPS : {mean_crps:.5f}")
print(f"Mean MAE  : {mean_mae:.5f}   (CRPS of a perfect point forecast = MAE)")
print(f"Spread skill gap  (MAE − CRPS) : {mean_mae - mean_crps:.5f}  "
      f"({'positive → spread helps' if mean_mae > mean_crps else 'negative → spread hurts'})")

crps_per_step = crps_grid.mean(axis=0)   # (seq_len,)
mae_per_step  = np.abs(gen_3d.mean(axis=1) - gt_close).mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(steps, crps_per_step, color="tab:purple", lw=1.4, label="CRPS")
axes[0].plot(steps, mae_per_step,  color="tab:blue",   lw=1.4, label="MAE (mean forecast)", ls="--")
axes[0].axhline(0, color="black", lw=0.6, ls=":")
axes[0].set_title("CRPS vs MAE per step")
axes[0].set_xlabel("step t"); axes[0].set_ylabel("score")
axes[0].legend(fontsize=9); axes[0].grid(True, linewidth=0.3)

# Per-window mean CRPS (bar chart — highlights which windows are hardest to predict)
crps_per_window = crps_grid.mean(axis=1)   # (n_windows,)
axes[1].bar(np.arange(n_windows), crps_per_window, color="tab:purple", alpha=0.7)
axes[1].axhline(mean_crps, color="red", lw=1.2, ls="--", label=f"mean={mean_crps:.4f}")
axes[1].set_title("Mean CRPS per window")
axes[1].set_xlabel("window index"); axes[1].set_ylabel("CRPS")
axes[1].legend(fontsize=9); axes[1].grid(axis="y", linewidth=0.3)

plt.suptitle(f"[{ANALYSIS_SPLIT}] CRPS — probabilistic forecast quality", y=1.02)
plt.tight_layout()
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13060\2338357758.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Mean CRPS : 0.06258
Mean MAE  : 0.08896   (CRPS of a perfect point forecast = MAE)
Spread skill gap  (MAE − CRPS) : 0.02638  (positive → spread helps)


## 11. Interval coverage — calibration check

For each nominal level $1-\alpha$ the empirical interval is $[q_{\alpha/2},\, q_{1-\alpha/2}]$ of the ensemble, computed per (window, step).

- **Empirical ≈ nominal** → well-calibrated uncertainty.  
- **Empirical < nominal** → overconfident (intervals too narrow).  
- **Empirical > nominal** → underconfident (intervals too wide).

In [60]:
COVERAGE_LEVELS = [0.50, 0.80, 0.90, 0.95]

rows = []
per_step_curves = {}

for level in COVERAGE_LEVELS:
    alpha = 1 - level
    # Empirical quantiles of the ensemble per (window, step)
    lo = np.quantile(gen_3d, alpha / 2,       axis=1)   # (n_windows, seq_len)
    hi = np.quantile(gen_3d, 1 - alpha / 2,   axis=1)   # (n_windows, seq_len)

    in_band          = (gt_close >= lo) & (gt_close <= hi)
    empirical        = float(in_band.mean())
    mean_width       = float((hi - lo).mean())
    per_step_curves[level] = in_band.mean(axis=0)        # (seq_len,) for the plot

    rows.append({
        "nominal"    : level,
        "empirical"  : round(empirical, 4),
        "mean width" : round(mean_width, 5),
        "gap"        : round(empirical - level, 4),
    })

df_cov = pd.DataFrame(rows)
print("Interval coverage  (empirical ≈ nominal → well-calibrated)")
print("  gap > 0 → underconfident   |   gap < 0 → overconfident\n")
display(df_cov.set_index("nominal").style.format(
    {"empirical": "{:.4f}", "mean width": "{:.5f}", "gap": "{:+.4f}"}
))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Calibration curve ─────────────────────────────────────────────────────────
nominals   = [r["nominal"]   for r in rows]
empiricals = [r["empirical"] for r in rows]
axes[0].plot([0, 1], [0, 1], "r--", lw=1.2, label="perfect calibration")
axes[0].plot(nominals, empiricals, "o-", color="tab:purple", lw=1.6, ms=7, label="model")
axes[0].set_xlabel("nominal coverage"); axes[0].set_ylabel("empirical coverage")
axes[0].set_title("Coverage calibration curve")
axes[0].legend(fontsize=9); axes[0].grid(True, linewidth=0.3)

# ── Per-step coverage for each level ─────────────────────────────────────────
for level, curve in per_step_curves.items():
    axes[1].plot(steps, curve, lw=1.2, label=f"{level:.0%}")
axes[1].set_xlabel("step t"); axes[1].set_ylabel("empirical coverage")
axes[1].set_title("Per-step interval coverage")
axes[1].legend(fontsize=9, title="nominal"); axes[1].grid(True, linewidth=0.3)

plt.suptitle(f"[{ANALYSIS_SPLIT}] Interval coverage — calibration check", y=1.02)
plt.tight_layout()
plt.show()

Interval coverage  (empirical ≈ nominal → well-calibrated)
  gap > 0 → underconfident   |   gap < 0 → overconfident



,empirical,mean width,gap
nominal,,,
0.500000,0.4383,0.13875,-0.0617
0.800000,0.7391,0.25328,-0.0609
0.900000,0.8309,0.31233,-0.0691
0.950000,0.8855,0.35462,-0.0645


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13060\576960551.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
